# irwkv / bux RWKV-7 Hybrid：G4 FP8 GPU 验证

这个 notebook 自包含当前尚未提交的 FP8 修改，不需要上传源码 ZIP，也不使用 Colab CLI。

1. 在 Colab 网页选择 **G4** GPU，并使用一个全新的运行时。
2. 从上到下运行全部单元。安装和 CUDA 编译可能需要较长时间，请不要重复点击。
3. 如果任何单元失败，不要修改或重跑安装单元；直接运行最后的“下载日志”单元。
4. 把下载的 `irwkv-fp8-validation-logs.zip` 发回。日志包含真实硬件、编译输出、测试计数和错误堆栈。

验证范围：在线 per-tensor FP8 权重、逐 token 动态 FP8 激活、融合 QKV 统一 scale、
CUDA scaled GEMM、BF16/FP8 融合 QKV cache 写入、完整 Hybrid 回归和小模型引擎 smoke。
这里使用 dummy 权重，只验证数值与执行路径，不验证训练后模型质量或吞吐。

SM90 使用 FA3，SM100 使用 FA4；G4 的 SM120 使用 pinned vLLM 自带的
Triton unified paged attention。三者都保持 FP8 Q/K/V、descale 和 BF16 输出。
SM120 会实际测试 Triton 2D prefill、3D split-softmax decode 和 combined FP8 KV 引擎路径，
不会再因 FA3/FA4 不支持 SM120 而跳过。


In [ ]:
from pathlib import Path
import base64
import hashlib
import os
import shlex
import shutil
import subprocess
import zlib

BASE = "ff5d41e1691d3d566264bc0b6ecec02ea854dec6"
EXPECTED_PATCH_SHA256 = "84cea86787cd6f19d3cc86b7b8801b20501b16313098a459904e25ca3352821b"
root = Path("/content/irwkv")
logs = Path("/content/irwkv-fp8-validation-logs")
patch_path = Path("/content/irwkv-fp8.patch")
assert not root.exists(), "请使用全新运行时，避免覆盖已有 /content/irwkv"
assert not logs.exists(), "请使用全新运行时，避免混入旧日志"
logs.mkdir(parents=True)

os.environ["PATH"] = (
    str(root / ".venv/bin") + os.pathsep
    + str(Path.home() / ".cargo/bin") + os.pathsep
    + str(Path.home() / ".local/bin") + os.pathsep
    + os.environ["PATH"]
)
os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = "0.0.0"

def run(args, name, env=None, cwd=None):
    merged = os.environ.copy()
    merged.update(env or {})
    command = [str(arg) for arg in args]
    workdir = Path(cwd) if cwd is not None else (root if root.exists() else Path("/content"))
    log_path = logs / (name + ".log")
    with log_path.open("w", encoding="utf-8") as log:
        header = "$ " + shlex.join(command) + "\n"
        print(header, end="")
        log.write(header)
        process = subprocess.Popen(
            command,
            cwd=workdir,
            env=merged,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f"{name} failed ({status}); 请运行最后的日志下载单元并把 ZIP 发回")

run(["git", "clone", "--no-tags", "--single-branch", "--branch", "main",
     "https://github.com/howard-hou/irwkv.git", str(root)], "clone", cwd="/content")
run(["git", "checkout", BASE], "checkout-base")
actual_base = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
assert actual_base == BASE, (actual_base, BASE)

encoded_patch = '''\
c-q~4`+M6slHl+9D|p(QT`8hyy(~Yv)!(G^nDlj$&Lruc-OKr7Xo<2pmPDDP9D8Q_f4_PFAOM1TIO*H{t{;{}0tKK@FBA%eSu&q{
-R@OVc+t^iyqaDwqwFRxzlPbDo4b?n`r$H5W?^&{uZm<EtRK9~gNiqj)hxdEdcEV}=<+xS;$a-krz5Z5>y1aFMz`BNu>D4-
(>b&cfBv)A?e|apvDd+me$Rs^vv}^Uqb!fJR@>|T;6>RL{S2OFSDPi&eMc|P8y^0%4x$;r6t7y%ZnsF&Mbr0+hjlzjR)voMqs^k2
40@we-
@A?%>q)anZ{k(%MIM&+00e#CWUFSo4(>9#?=I6>yy!0Cm9v52;ADJSTmNdk=`Q1Cnmu$kMY2f#5fw?Q0L>R^q=D)MXYEGE4d`|g0
RmO^>2;b+<9za|IbWYPFWlAmA^rmZZtl9%XnGxYXV|m>BH9!wJZN50fBIwmPGHNfaTx}<6<MpzvU?d7(9o>Q_;zBgKOUZrY5^5%c
ct12c^uE2ReEivx6ujF+i~ANMRJSsht-sbF59eH(RoR4Z=+?LucK*9WcW5+#e(MKKQPFUC?W)km_&i`+c(R`tLSnOhwCh!C-
)(uKUrN(K4zQPDhOl}GMR)LnE-V63%{IM{70t^sH6Y1O5WU2FXE7h0Lq#V{OV9)KEph|%_s1q6$R6DHQi(xF!@8frnH#-
kY`_DsQSXHUBII>3t<wX#R8C-
C;!0y6>Fa+aB$Mc2^^f9@C4%2h52R)lIFoZe?DOUDuL<qp5kP`e*FgMBQ2sue(rswPhf#_52yuanmd!lC=I-
SmR)NW(Nz<`2k@!&xJffkFX)?w=BB}^SfXY=nkMe4Xzg?~HN-
>7ss1XD^Bh3u1VR4rp4f!|!VJq~>1H)+!RQAtgEL+8O!I7dL}L3WTEJ{&X`YAhGnlT|Amz8c7Mf*xd@{c5M?r9Wc04+b2Bv1&3V=
1q)*?g*qY<i%4t)-
g6!KzreqL<Wi}?E@&8FAq=O2ldaJsdhpi_zhl~B4uh6HK>ECW(zaZ5wfI?a=U@E5ISUcN|+a2c)Fz_#9UlNa7)>;WE&<Z6>{a_`x
%&z~X%4F^Z?cRcbw{dm!Ae~0jX$3Gu`dH?KH`0Uk-XFs<ZU2cNrH%Rhux|v0w?Edv%#iM*0Eut*MiHFCNiR}K(k1ts)Ls3aRRUkT
_xAq12pZ{ryhaJ_Z)@c}zmc#jq+N{)_B}=X|(#8dx-0IWQSJu-ZJ?#%poxt(}EC*G{syb+UKX^S~Vzo)H)Ev>o{P+@v`01Vh%Kre
Qa(dx?Ns4QI`li>>Y85YXoU+b#I7ctvopxW0-VFB!Sc(%igi3O*qeGrdiHBR?yYam{-
%}*%8}#TiHn64N_hsYFRu0<;CLW#j`u$li2>S8K^kh8Tmx(t!d3ZxCJoeAL&e-
qwfqNU?za*>aVl#`q@1Fvr9w82n5be(Zt?_*kT>qe#nf7~Skr(&Vc#WvB3qgVKa^4@~^N#f#3>&LB(f1Lw1#&_HGv1?YF#sWNghd
u5MNVZ`Qt6=TpfL_degDMk3`axwfCnMW@g103VF+L6&|sXb7NDKrHyK?Zl>y{5U9Q36K+{c7!8sO3YbRT^;M;Puz;Aca0!#>?yc|
HZeXo48juH@j^q48bYjlfvxn4wt=r{J&dnah>Bil^$>;n+jM;NA~_%oz5z3i6y(3z`6dKoQfsEcR=>w)kRbxt0Jt((6^**bq0XFo
2|=?y3;dhqc&i=$cI_U_VT)({Lc-{fE{)H2IWoB^u|L;(*1!-nDNE|f;>6Zo0MC_|<NbmcZ&7nv{S=8Wd%xX*L*M>KcG&1t0-
%B+%#y(={cp;l>>0HP#GzHNPAOM=RhbmLj9FXZh4m~M*orU-8jD}JLDCoLHkFdJk9hAiTFT1LG-
Y4%ZX==UErG0EI(T@qb=@AcZ!jN>PriHA1m8Wy6tP|)2<_gVN)N~CxlQT>U02K7a+y$+&PBK6R!g1@hU?o|o=p-
On^e4fXJdP%aMUw(&Or>CEU#<}oNO_R)PWnkPEfU%plR_^8%!2SS=698*<?i2vhfPg?Ez4(5ec`Y)mz{bddyTiNhr4TT1_5-
51qjdJJ{%9rl>(TEXsMEN%4eHDEt_Z0+KvXE+wRNv*8@v7XxA4Q$k1v0E`_so4;x}say}Grlp-
^FDK%cF09POQL|IUKiW0jzo1a!nAoF#WH`9O$$`4e6xVA$deApabX$NuQp>l~jP`^UqQe-
1G`SnxmtUcn+6hQ3u~e`nSripNSh;60hbVkR%XhZjGz)L`O8jH`CAW<(QhHkQ!A0M+2p66#0A`8oFL{QN2|p3%amK$ZW23bdHCsS
0ZtsPgGwFIx0OO4JbDP%)@O2F<n;K0%C;^xWiV*el5lKmZ{NvV1Kd+@P#Ml36dU0~aL^5MJMF>Qrdat|YAQzyF?*s_^B5*((tB!w
)~S%A&l21#HI|G*dA0U8?Gb)@?A0iU`Hj_lRk^Ao!uxZu^e{c=N3R-
t7%gDbTWp!Y35BO6w+cS%I<M+SymTmtxiP*rD6tq=NnC5ci*+Te<A&)xPQ5Z*FSq+|<^&tF3caTgO-y_O`y=UUyKTii)_>SDJ^E?
(Hu9$}S;AB8!gS=`Y$cC1uyH$$vuUgOX=h$sq7O>v=+ow#b(e7XSR+_taO`QJSm`rLChRE?X$2aP5?$lC1_^RvLwc?)@4`E_Cogv
fQY%Lz;`ShsbkjDplpW=5E<7*$vn2w~_6S19|i9LEaqza;Jp1%I}i#yLv2adRx5jHQL9B^hd%xWL6BZUiYqFeRs2^-
p!VJcU$V+ZK+og@!Q_-sy}d$#x6mBK>6N2<zH3HXyTsU{4PEZ75M_}BZR){&`z;W{WI2JwbgJ<VpMCGwYvw@{WfD_s0WWpt5!EAT
D8d~^S<tO)k|HCwhZQ`wkk;b7HZvvi|+RMap9q>hVL2%tck-
M`huR0LX8FgJnQ#DT)%}szj*)l#q03((_i1Jtr5dp{!b|}edRy`OdAgFSvt*+Xx}YA(xVK5<;=U>_EtE!kK_1qcsUA!v-
xaxJUTNS+}rB2gL_*E+M_!22e?Ib<`2iX4li&=Aub}1eZYuObR?tgahXJU%$rTCG+Rb;hc_S>^uMtAM6cufc)Gz?=+gaw8>yQ`oO
RPLt9Vv!ZF%J2@^BJj-
T?Xe=(oUo0bl^Zyn|T4%ss@JcO5~)jc#tiLI|{}5hW|^B0(W{otIxH^MZQw^4)Nt0YtGE2aWDW?CdOBgZlEWA{cj;>-
!^A%$uhfp(F=EI4z*X63%5bg?47xM5Fs9O9~iE!jHXZj!<J93GE~Y-aFb??~-
#l${xHTi&k>`ZknziO29BmZ2CLzDqh7|G8L{lD3l27XK@}ESnUoPxlD0ihft3^mcz&Z=bQC9&9J{Y)Qis<F<;{3>bl5}e*VpOBE|
PUzMH&zH|b-*#b$Mb2*z$N7O)5?%ZhC$ps5@PVZ~Dp)Nr*5coV9e;SCbZ-RswHyk0OI1igRDY06+uo(=$MxFxknzrb{6*fW5<$)g
1kniwd$bzGC4d$R<3R4g8R+Svm3Yjl?|hAct4L|TZMj&3(`Ui=$kYq?&;Fr`2L=3OU9bPs_?G@yFRh(>)WH`N+la$F^si`ct-
@cLsI;k#c&004aCFbyCJfcm`1d-
w1WMVt5G`Mam^bROdlcnsYwu3=LCs_(1=`cFNahFOC8|8j#<rMSkI^NBP9U<eoK9OW(`I*V~K{!Zi=c<<%3Fx3c3#_<y)0;)1OiP
+d0sS#!ye}b&c(%EJT@Q{+wqZ>3jhp{VSMZ~Xw49q;W^@#6qVj7*#c07*<qW5z^D1gbyfU14(CVt>4C%<8igaa4_jv*3`0R8FQ>j
%9?hxWQ75WhHDSYaV|Qj37BtC+zujcp~vmrsWt$fwSSH)pVrdHHk%zq}7`K)U+WkB`96SaX%k69X0(8}yj&yuTjdNP!iwlei&;l)
*zG9I;+tN~3y258NZ{`rx_83o#Nka;C64ylr%b&%J!TNMLr;d9jS{^};c*98o4O=vdMqGE0$-
V}`*}!;MZJU!gJAoq`Z5gh)gA%{?uLNK$Zx(*XFznNU2-(z(%bif8Z9(pd*6gddC^8m1pP(mY6-
LXZsc5z@Xxf;T#k6ycsrf;T#k6XBjqf@9bA72zFX8$B3@m_`I?<v@hweP08rg)CBx$3otSvmzDMnLVtcC3J==(Ig-
`BOrl0E@d_cFtLpYi8B2=Q%WwhGkF#ooxg%8Enj*`UQQ155JVqw2x<ssVb+Qf&^I~};OTXgfqX}?21EWEYbPSO&k@r4kvAH_pb1L
kErE!Zo24cW;%<NN9jhGTCweoZVg2RBn>URP)zsKf@)bxbV4amvH87~<Wjvcnp;;iqzkbuD{Vt(1aYi2+9gy<~j%cA<4i;u^db3V
Jpq8{6O>x5?$rStZx6`Mmq>MmoAwQm;K_42ORSe4SJJd~=$x3uxw#dwZYC=}@SXc=u1rtPkP1NF6_)tVpiU!zM?`ZxN*GsFQ!i-
zY0zu2$ECZ0(^`^nW%D>U6(f;0}ls|NCi}v^SR{rrl2<d53G&+dWS!{FPHrpU4VfVm<vst3R05lMISdJ4ZdHLoNbat^3;_NTb4WS
4?UnVm)!@cKQj0sRpHP(dpC}RTTNsTq(mBs|D3qJq!{%=437Sf+r|MBDdpPq;BUp)Ku{reYhKZZX(efIN<x6dc@MU-FT>SXX4r3i
AUFe(D?LtGTt0}#wb3h>kx;b-
XhdJ#=o96pe$hGhxL8<uZflyrgyq>dzsfbfwT#a1MS_?!&jFZ_IoJ;BZngt<e~=+kfh&ljJ4FHa?$RUFTd#!>$v^?}xbQBz0`^cW
Ql>2IZBF5@|xO&sQlY?T$8HDUE~lPp*$eYW>N(=`mTjP*fu5)%pBkb3_73`jd!5Cyz}o&|4zdHX`h%=u=q=!y<g#lK|H`Z{OD$z4
HH5(Nh$9WW29_xa_ouV07HUp)Ke`HS$Ex3B->^LNTD$>JqYd(M;qLQt9of|--M7)cWrFo<-
d?*+616wU)B{wi33%{D|<G=sp38rpI3iM~P*!+<)LgZk<NoeQ2avjC^Utvy2-
pJO<}6fA;G24ahu6W$=tT;bT^5m}ZfmcDx6rYvi8<oLaBF?`_Ha{S)6A3iBxcn5+lq5&EP9oRv*q7jDAQ%RAM`e!K#_(C?`*fxQI
_c>ls7+FZ@qnPr~3Z>Cq1mue{7P~o%FE>|L=r?M3Z&UB56-
bl<)mY|zCdW}QC3+_SIWPgri|HatmLR`~pHEIP*0yst5`j4^e?GqW@G*S(`ss&PLJCd#UT_z$?v6n59<3ihhpfEr(t~d(ExO%R3M
@URAOEL8G2bAMbw|n(J~0lC;2D?hEr_VkpV99@OeXXKkp%&p!)SB#k|D<ny*^{^l(GUwv`li`?g_k~3esJGV<0dm#GF7P#ApwAA5
qf?hz+Qr6eHONUAE@QKA@qBT_<gJks1{}_@#wMTv~idrP76W(G^Z`Ax0`y9+-aV=IYv5>M&V>0%O6(<7#x-
I+k7Jvn_0bK&OuQqykb6ZcEWxBgs>ighymwAE8x#w2sKM^AX-^4oicqRLdB`5A;f{z?vu*2%4ot-SPc3*d$;d$Dm%OaaJVrhteQv
bfE|Fa?Pv443`7QZK9iC0l=~icpIn~akz*Da=`@8Qg!kOCkKsbFyg)zdAI_|*YOk<tW#KPkgtw0DC}aiiFx^Yde-sTi1*&7?_M0S
-Nq{*U#<3rN3bkD;)O8SU6m#o)>-
d}4ALX9Di%v&Pab(;0VWh5djYVQ2rz|~ga@%>OucvL%!6gVZR&L^n2r>XNh2^E!OCSdi`T#^v?ko#gZ1-
(i}`rDSqOg(A*+iE9vmIP6t=nWMX;q+PhhzEJCwq_Ed<u+986(gKD-
1l`e5^P19Dji1VAB<VS((7WVcRMs67B7K!Iooi!lLuLCF31>%WJuUWOlj`StxXFgU;zNnqkWV>~s~I088Z+V$vq{#n@SrN0g41cw
AhE*U6192DJWXoPk8{h>H$Cw(4i7EKpK;X^dba`HmM#1-
jclHs30+&vav457vVox&$U5uPYEf#yh5f^IZx_k!NYYo(WYoZZ2Ap+#C}##si~f|sYqXTy_Yq;@t;VK#9s3eyG<PljSKjfC6g5x#
oq@$dz@vOA<Vvh|@?cet*Y6Gs=g0P2EWe$$o9o(3{k^q|O;76-
I8qBSr6K&+uJZ;HxyHRX{aHC@K>O2|NVkHFL?AV+x(L0S@F#P_i1XKa=04V?etf#o^QIO^QF@f4?*oondN>^uS}V@Ibygr~R+SOn
M6x#@_=!A`*An84}Sq$y=#kqz7iC@>dA+%8gl_b=#z!L<yK!V)c=DvxFxv~}`poG>A$zJO?<_0c@Z^5VSFIS>rtp$3*m7Q!WH)P~
ap!2_RpzyxI27lNqb*%ANl=1ZVFFi+^zVDH!;_JDf^C;o_ba=y*T!Su0CmU{i(VC2=Epu`h0(gf=KvU0mDeBsDq&li##5<<=^y<F
Kii-
&Xagb8eQN^B*F7vikA7Aqa>IXj%?mZewqsd+B%G_{FMBfXZAlwCT!KE*&|4^=kyJYBOF@XIw=ro!%44$soVhU<CT{aM+U>d;==(+
!YA_HViO6b~b-
SwZ4C=*;yRwIN`MI~@tAxRI76FqEToF$?eVa20<!(rU@(e0HyrLxssBS@NO@yKc}^qYKL+c0SN@Tp4ruxm;?1!UtvM@hRtZhDc%e
Ff4Js{%ANHgPsM2M8}*FSNnF79fafvp1C?|dS}FBeS;2q$FR~6o<*kqBYfi%1U|Se&X0QXsN-btL^~bnqq31LG(0+<?enTjXq82>
1Y4g@AR`dyB$`;8<?_MU(jymL-
}i)NBu?*w0EG3!mnd6;39#1v&Sp6vsp1Rj4y|h_8vsZD>%%W^J+V^8bv=*b=o+v9ZGL?D5nKM{Bhv#i(M$|E!0m>b(!N<#d1aM{9
=bPWdqYl^AvJNK_$8U4PFi`x;o#I89-odpz)|lzu&hvKeVt(apA5a*<xl=u2e#zLU^_qZLei^mO-Fy<ANV65*0N)N2;X{8nc^9`-
F%UHAnCi^BD(TO5O(qGLf3@$gJ=8>{tm|7f@2{b34)50aK085ov%TK7{^}1ye#e*HoV_4YCMC-
g9g%C#cFx~ezP*S5?ERZVn&ye*qIV`BPk5DQAk=Xk;kQqXmpv4k6<}r>IB6IzF3bNGMC-
&QVx>)HqBre4z2iPJOk^;N5aOJCVVb;m#5J>nxc6|{$R;x-
FKVIMUoSD!V)en1P^5C7j}mJIPCR!4DQVeVRwqr!=gJ{q1^^DM7W2G>qr}z)cRt_f}J|$E{gQ?izNyg4FRiVY;BRIYkgP6wL9YNn
y5x+znOa69Px&=JxAWxOi}0Sr%1;&^spY;yu+RXP=rovP16DwTe7$kr5!Edp+J>banO0N08SRXph*FF`8wsLxD*2dfjY(-
9a=Hb(k8~>09xCKh@;=s<_FFP;<YvC4aVJmzuU)*9US4*Kf`6^;8>(%NO-Xnd0JPOag@Wl5d9Uf%-
m$S?V+{|peMW|1S?nQG@DZBOhFT%bON(Uw0=0k6b=A-
B?Xh*5AibEPC*7(>2{yo+o5==2$$c1@*fJfiD9|ZDv5B04jgQj<0K;8QIrbL(Gpq*k$_tRi)RETFr{V^FMup*Uuu<pVNX69Io%md
8Erg7xVZyZ60Z1~J-
fhr1`E$i<Y%xocz*=DaF=8S&<6|!zNM=r`B*@2E3ZJKLyhlh*xSEdq{4j~Muva)^y4csDBdQkzeheu3v|+okOAQVAu}cU2L<(Gv;
en7<^&khOHb_lG-
#&jMC6xiAXnPiLGqr1T?w`R_fPNCZslK+LOIHLSqwB8U#97eh~9|vhKzv_%OYc&fUt!ta$u<q_EGE!S)`Jz1Q4?3%*aGifJa=O9o
w9w$@yGFBc2Y<hG$Vf2&R|w=(2C+%y8G_I7Ih5TDhL~DBlfy5PyQO03Oo_RwT9QYDs@+hH!sk5h1lp`?b_6B%&!GNL8k~O=N#s@L
m(+&@MQ7m1Iq;{BY286>2s`ej%Ng&e7TA{U;^j@h0@ag`o5=?0_^_59a?m6=5v61{TI>&Rh987#bM!@tGRqfNdad$^rw>d;AFAy@
!DSebM3oyDBwpa+xfI7lfQ^r*z8y>~zT444imopp>u%NI985gnEx#V4%-
kxrUNC+U|4CFB3)?M%q?ojd&&nFvBp9h(!s|>TehV_5!_cttL`?so^)PGUh^G{shgo8MwX+P_(MTDuXx-aB^(M)M5XDw^}#`k$|d
wzE>C2?9!gK=9kvt@9w~4!mSDqBfLheCk040^l$zTc>Nyj?pH)pFVpN9NzB)8nmfDsi@Q%H@(P1qv9Augq5KuJ=~zoO|3ceTv?nb
$EeWx?ie;fFw-
svPOzdN4Y$71~y!XF6Te?IVE6pP?jVKRQu8TFtJ0aTU!@)d0od&^Ke=zQWwswd%S8ZLqx!zN{fMZG_haV>;>%E}TPqV9BAO-CmOt
=9ir<5hH;n7Qaf+v;nmr&wbS(S$Yo*UICI(G*~0Wo2?MmpzG&p+aN$*e-
}cM7^X+oIt}<8{ta;<7RNA^TUBs){?k4Se3H<~MT{H44yFoDL~D*D0(dw@rnXjIPMQB<P|gRm5GS!KUv$dBRX{z<j^TKQ*aL$Ynw
*b5X)+mtVQo71*8Cyl^ttoei6xUuY6x8x$+|o(;@+eX(cHM8-NB7Cqh<#`Romzjkr|zya-gF7!i41@?l=_QT154TzJ$esdZv(3G5
oi*y<-LRgn15w$sCIa>49cLU)9pn!2R(S%^UvZ`}VJ1Q>Hr`2c4t4Mzf7S00~!f)0~xu)a-
@|RVJ&V!mIjYqyq%<`;ooLQR5&<LUTHIcicoDjc**>T*PT?WDE^7L{RpSr~FE=am4-
c^=XhJ%scKZd0td=50qacfQ%DXrf~zdc>eeqDp{E`(RxA~B74?N?QWYuSJ04yLjdIR5haGx&i2`=D{Vnssmm>xiPrtn3PGV=J!m&
18S#Dexhgsj6mMx6aUs!so>_*{p0IvpJEN(|TBD=``#a+2cd{*f#CP&Otcc)9|y*F7Dpf8-8^LzHI^uYu4&T{S4G5Lw$Ugc*m-
YJf>##4$)ecSS-NITWC$>VzWh3ZYW<gI%Pd{W$}Up6AxJh10V>QbgV4cGVx%YF?URkG>XlF<d?A<ROtw-Wm{TTf#f73Xph0H_&er
o)So0P%x?3&*E;pRv$o0q+t)beYhs(-5<y@6P_NZonc#M=i&)G#V$>rMC?rEpadMhlXQWD~f9QnR^P$+{%m?KG{}0+4u-
z=7VL~Evhi|#rHkQCGQ_vHW$aOG$lkwM%Ww2RMNnNNU;<S_16=j(<Dv=7K#BiO#c*ya~-$H>YnX`<Rmr=N%<k?Tn34-
SEY~BR75M2UvbckYT84-W-`s1D612LuQ_>$I43CRlb-|0%QkH`bR8ts%G;eA>&p_zjwZ0l5{BE#eZk+fMH%Lqk`-
k}zU>Zf(QL!SekBJzA`P{pKlmN8MD4*gR+Q+)F6h|&}^<SZbB=0UW^XrdM_^~8&Iwc50dl9#wcaYGJ@(11sI@wDG1tVMc-
(cj5xp4!HbhD`FzmJ~<2wDjAGU76uP9ucz$pa#;Pa%-
buJ4ll{Nq}?wH!96@fY!9u8%;CX6NO;YZF?xk6P?6w;t)|6H7hHME%?S>_{REp|Mi6(aD915X|dLbho{5gD1x=dS^wmCINpD)VZ+
<K;;?Jbigk=KoBqO2&7!*N2WpxF5x$)VMyP@?wS=zGcuJk!`H|}Zf&sC60mmFcD+w7ven3L{fDw~j6M%B&P}Y|@W|0-
uUTzkPrxH9>?L4}w4+@x$OQS0Xor{gC2qDi=YqbiX3`j)3^6mrYcJMs~`ZY$!{ut#k$Q8=1N}ix$U*C54b}eYjb68GeKF@2Bq|gx
QOf}+%S_x74GIJBvv>=9f+=M->p*$)w@jhKx#j^M+$pPWv<pyI;TPl-
#Q{)^EFv9o&^Vi_X&>Rb|ipk3e*jqRI(Z0fcank;`A;oy30dP}(y%S$;Q_I;81SWd`#!QY0avUAd1PX^?S?F`f8Ijfodi*8IW}%9
%(iE!*e@a#o>@JNtZ$95(3>gK4k-
@^D*8gNkzc=~nhH)SFRytse^@o2bO{r{Cl&2NcREVMO@ob{2Y5S#Lt*={nNQslG;b#eJYf*vmHIr?YzvhUdlqnTvx2)#URGQc}pV
bRw5g-KOjfu%*R!PoPiy+4%%R|9_I3`1BdRn?sFEZED$|EJ=_(>CP1p(cpg(~A%6K(-0+VI_r_u<DEZ$JF<-
hikV!$W<R#^hbSlVemRCucP>Of$4}s(TgXKdn}A_JNQ0{*-
fRNnx~)19Cbk!jN`veD8^G@STh_oHPx?AC{+hDNdj&w=HQTmOgvQz;dF>y+x?0jgCD1Ee)_nn9|3ywXGfAgT34JRdW;hr806PW|Y
us8eW5>vxvVUDXJW6%!0TtW=QVu(dlk$`#>T<>AET65G>!M$OS1}Cm>8P9X6(B&}Uz8I1P8foY*o8Gyfn}Fp}ver7WGsm~@|=86h
PJ2*sk8_YTipQxa^i*lu*nanNR&C?`fCe(C+kj8!r2nu$yVZ|HZ+UYsXam{NHaf5B0K-
knYUS!=%TuvhC0)bTiu&f*g=Q2XO}G#b<zsC!tXH74kYX-32Xkf}x@Od4gYRYQ9ktdxB?->jzM_)e62FRfC#iHRp#q-
T)mFQgEjkV1Hn1VWU4&p-
ahyBD5F3uX#fQR2<he=pzD>Z`2S#w|Ws?x(lg$|VTzufINh`!Rg>>eshFhc8~gc=G{j^vB*4KGeV;NI4Df&wOm2g&`3TEycolG-w
F}azMMf!yz8LkoWvlOX|Eq2i?is4)3CD0i1#8Bx&9e4R%T}n4*k9DXg|6GqB4`4j0!}bxzbzy$hAIPt?MDOwKKz?%iUsG}j2}Iu8
n34~8NQPT3DRv6UCP&sPk(QpVR}V~Xs--4pl|mgovJ%vBZ;mP^KsWJ@I{Im}NeBDX>na-fxJ%MfKe^x^$PO_4Q-
bWexFw2rgI_#Oismrg%X6a9)}4d?I>;c(03L|Re`u};&pQk$_vJ>aKe6hI9_W0KF<kQiChZ}>^uo_yj)NJ~6vA3WGvZy2Xu@?4ph
pfAD$A=4~1D-*Jnm3$S9*FiR}GO^08pjfk9rb>Ep<YB;Ql~sBhoqI1wy*@7=kCFSqq8y)?++S@%bXlx+fUOd%36=l^0-
0VK5h3UwzI~!~e=e%?4N(+NLzlgL(jWUqp$jo*d?0t?weEvi3_9z&r8IF6UFP@~Z%pg<F%MU~o+V2<qu=(9JSl+N0hkrbbqG>an`
_FzBQM(1!$e@>sX5UK0HQt!C@m?jeGPtLh-
+i2%al)La;Fw77e&r3{I`m_4O*a7!i_ua!MV}e{Q;fCe=GLZYf77hrWSvbtm81N6QJ4{4-
n(I>vU6SzfgT@b|tTebxX*$wRgap;Gec-NoiKH{A!h`_z&1{Qmg(wd<c)V^@Io9>?lYGdcYt8wc0GF#op+lSuLN5{8$qcj9?3E@M
P>sAGhA3R!;Jkc+LET?9|q%-^E76JZ(U5ejTkbL2z~PGE)kk5F9R&n^-jDi*;#*XZR|SFSyjeUI0%!jGm(-
IUH*IQ@4L%IeREY4XO!>qHXnjvqJB}r&erV5A7fzau;BRv<QeVV0E~HGtecTeWIFpeTid(MT}XL`C6&saAO#>fVoRbewaB+g>v4N
43ac1{7!<w_Y^&PoTowA|6?a*doKgd)7HB8bpYar&Gk5Y(CvGk$MVHJJ(R?Vs=!4V3MsxK{91)&?iJt59bRh?Vk@2Y&-
XEBQH9|%Yc7>>^SyUUyY<QlI;H&*hX1{HW)p0UJ@fc|aattunNT?#CYD*aTsmmGL}~S%P7j468@v!dVKkx7+V_`}3h6q^l9o_J<0
ybiMGIUjrK4ZVSZIJ+7+2ZR03%3stx~;9#(p9h8`#%ss$CCEN%oeGHw~tZ1A<?J^|Hm<u^BV1xX>7$@93bLF$FV=bfKFSn73#-
qu^ymJZDZstF~%!ppW=%nk~ubkUw-!jwO|>?S93a#4GouRyDnvyy_jD(A8)C;fc7^^nmU-
w{l*mU*1Qno6<KWLuMQX6P<LUNM9c{(~bCAp0U?sliBb&_b$_P;WWZb(f$?CZ9`4i#M@n?rhjvmep#`5P}q;u8SaO$;DkYC*4wO&
T3lSzt+o3^V^yQoXq4J3hh_tzma8BtJ<uVTOr|9tooHCr&l4P~PH7uv==V?Y2JK_~bW*5;<lMyhCb^HZy-0H^V-
`AF3pi~0z5M5V#&hCHH@R@EqQzdn)yS2=+8d9oQXG6kl=DvJC~E1({8-
Fk+a|coMWs5yImz_qQMK+KRV#f|tuGce3lb6aV4aT_Y_BkMofgMd?PzChmKm<PS6l1$2dB_<o?I={1jF#{tywiGwcIQ|cCho!*Mq
qGdJyT?gE(v=>){+SrS(uAF}d{u^CrC6#<P-
P?Ml(%m9p(ES^KSkjdok)A?i~Sud}*<#Hv8`Ok>eJW`T+!5cp|Xfn7J)#nz29?EpD#&2qvujI*C=7phwYRt?6zLHIoMEWgc)*3C_
j!~A412<m*Zbhynr>=?5eAZt_DR+beh2J-lpYAo6Nz-hsn<C8dS^#NQf^15EXU7|E<?}by>tQ}qr#52_F-
hll;$cV&V73lGi*BPM2IWm2%e8<`C&0UK!?fD`#YOOj~p$Gq8sB}|}02W+|ZwQFgl^<)Mj-zU=+|@wX!p^hd$aR-kXyGi;rm8hxj
*4`ee^<2wHN9)F`>2@8FuA{f=HMbexm_Bmcy({~fDRaSCzjYE?4JPbjxj*{q|O#~lcbw`pYtJDT2mcB!9FG3yQk&M9@JbG{=+Go@
;ihSOoev}Su>;7kz!Wjbv^-scWjd#V7<`7<-Qi&5>-@H8#w1ohzMLo^WOcRyb-RPhrIVzQGAs^l^u|shJ{$Rn+=+;x3%u%-
yQ2pfwGZPnCw`|YL#~JX3d(i+WK~*t9E@`?Me-&JJy~)X3+?VIPe(rg0fcY96>%*^mEH%t&*I4C%w1srP*{gU>$cl#{0Nja-
x>^YZk5)r1alF%h9-FykR30`QC^&)SVlGD1*Cn8eMXrpF?ig07aVeN<38^#~RpEg04+_9Z9**p@rXy#<`(-
hIIq3W!t^mbJ=Tb3~Ui>RlFHe?y);@*y$x98G9~cu;6Fjvi<2OmU7IijEPxV%W9abxhHC!=!3$C#C4er%Dk1JZRucC@jefU6!Qt4
DYcllLaHqVQ%HCXf;x*vqSkoTFsB>Ev@art@v;y~kQ)_oOQU2JQiSWK&JaFM>LZ9nl(5c#i?k41ej(pFfV&SA7A{lK*e8q1_y$g`
VXeIQ29ODVhU+w41T>J4voxY(8h$5J4C~Xk<xLL6EIYJKMBy5Nyn?uZ$q6Y0Lgm2<=j-
7hN&90C77Q^gBV=pdu_~i7TaP+^o0jQ`wM@|uFpCsU82!$<$mrZZKhzH{Td7Y>>qT^}?2v?@!KYDJ9Z{1m6G@d2=#{wOaY&kFc-
9wbwfChd3eS=xq}r&CQ>sj54P!4_na`Q5VJYW(cRnrd?6a`)9*YTwvN-
0MOBe*k*}3=<9t>l0Hnf#A>}DU^NR=*j@RVb3P}|tg+STz_B8mceu7MkHpiHp`F|FcK-
+Lm)EiWsrR&7^>^3D$I)s*8zp0VN8aJ^#2O;&hjb}QS;9&sF)BWPN6hXc0;1IorD4`A)X4@B3XRLVe1);YJ5AWBYEOTdbXk$2IvY
ek&HX*iBPs3R4;^*6Y1AGmM?zi66=mdqlD;lw_id~59ZDdtA?fp;=Kpc0`tQNG1M4#|CJnN8T?aDc~NIwO^Vd|yH1@cUInn{lV`A
|*?pdT;~@Za&ZXEEK~yI?@p9i-
&*Go>lA@K(WI*!D+69J7_^>H*LTE<(<`8n!iu7%jc@s@O?bOAeftd%a2FrSpKf;u(*j>hArLNK0xOPZ?_HY);yMcTZVE}4kOJJan
f;jOi_s;NCVsH7(X!e`+VVNC|QGG1a#$P3q9=U>g@I{!B{UsZ>^G*aO~FBKn?b7-
!j|D2rC_*z6ki3+`7Y1IGGW^`{(a_l9d(|U54>)3IDD_5Erh5GPI|KGO*;E_!fCN2kI2#`ea1(RC*2)yl}XZiAX}xmPkp&<bfHrK
}fWWez;;)s2LN)%_BhEnIP^C1ktYgNe-J9jhcY4E7L5CE2FOh+X%O>KNhQ>f$h{ySg@h<@--
9o1r&1ng^#|G6CPNp2CJBhcp9zoGL*U!KuLISXMT<O_=nFY4kOquBTe8Mkhtsx(>D|sy@cX|+FXHlSEI#Z*=QY<rhd+#LIzv#Ac2
I;`p$t?i|rVzQ3Jm0gbH5{0SWY<Z<afua<LtN_dT!pX!epx-
^vpQP10gAcHha04#V!&ai+D@E!B6LvJtl6=!{}TMkgnAezz^Fqta@ld;KO#>S_-iz2MVO<o|TMRntW>ZajMOq}vOQ{i?NIB5yMAo
eK@2yByQ><OxrKUz@kgVWXE_*s6g$e}s0ewOVd*>r;o)HoY%d`UN*1IT9K3B8XcV+omhV9QZ$ywWdO`Zti>5gHQeQ3+<t}>v2owx
6+E17>U>k$W<Mqub5YcNvsjsCmLnhfMN7rNPF4z%<wh1(q8*S;K{Zu!RTi4h2>s3wEC855?m{6K>#VQ=&jm;4nCqT*vAW7c=2J^6
Voncyx~}gf1)DiP@H4W9Va>O`bSCz9mxY(wFf)*DlN`=4#)k|%kfDN4Euv#-0!*0dF}^KKlZs_jq(5^{M9o;>w5@-
)YLI1!@Xq2;#rBWRen~j5}TqjPiwy+MY82jI``5`yph<IO4~Lt7?`7Aol#I*!J*N&4zS}=>Ic?fWxO4a?#&&6svj}Me4Mq^uA%kf
lcUZ>TgCCh<R1_G<CCqE|JXLHbS0ZS=9DGse%J|x&RUY!%o1c|qvX7C`s?62%0s*kFI3O-
ma%nWZVQIuI4e6*v`F2u1kcTF9*#$fW+>gdeJn}a<xn<!PRVQDVD=15ZDDV6JT`aYH^`q@g93+vZ4Aba1h<Z7S5{4Q3zVL$PdSTr
lnu0#$+ZFZ@`Q{whI$Nh0`6s0TyrjNh3Hvw=X<O89?(bnBAtf=8{2xojfoIF>xc`zwtzxGH#kLwtYB7_j5Q46u$%2nWxI0W)Z{pG
%toHabWVS3soEi$$3km+T({haA0`W6npuPa-gtuU9(A{~u|1abT(M38(1Cwfg&NLpRhSJS%gh+)CBy9k3YvpSc3^?h3)D=0+dd3+
PyOc2$8b24&f*FhYG}wgh9A7%dEH!Y!P0o-
*rIiG5KkGp=M|}<zF&S7THSgSdA3z4?e7``%twE`2f?WPKq65RMirRy!4&a<s^FwNhb3XHfgC3sYcN<DbG;))WIn3#-
E2ctO(h#GE<`$>Rq`bb2d+}Q*5MC-D_1zMiq|IYM5%*dwD8mIDEY4%7+Zv{g|!<C=G_}~<B$s4SuWERCrBurK~L}o%S<K8)U)4Ex
L&mmr<tNud3d+%@c5547A=<w8j0nv0BrYG+q!p#{oCbvS;icc91iSFB~g~7s;I^g)wBr9LbFkIPyTKLq*}kmpI-5%y5+osz-
pbQ2Q#_L*I6sVcGX9hrkh50)fAq~T5j0RL_2jK=a8;bfX-Uqd#vR%c;Pq(o7IRUbv9lBh^BZ2nUFx%Fq}HM)<$s<*LL6k7Bbl?^$
#M7;4X>3Z1K`}`+ogPV;j6`U2m<qbqHHkahdVJ|B9V~dTQSTB|As12KW^{+{IP@Mg7#Tbt#3ULo2i@;b<8UF9+-
_0}fK4hj>#<I9+doB3)nzXuECaBHkc}qqLt?D)KJCX`e!*`r~n3;?8fX-}G1p=#u`^cLd>uBCKX2!)vS%Efg-Hyef3!5Ioj2Jb7X
X62DQCfLss{bwY$M>UVHdwxRJ}RZ8Q~D&?o|5{3BCh&8E(QfX#NUErL2KMTvobG#v_RsMFwR8>Z$BJd<aL_*z~;HuJ^;+x(@Y1+@
(E2e7y4s2`6LRq{3q-
m2NvV7Gesurt<bgW%T|7h5a{oa7@b{y9vDE(@_+J4GQ<ic**52W%{Y0g6PnlehITzkFBg^WS2ISz#0ILvR{a%9J(TL4eOSO4+j`=
6eNFP{#Y=9WVd=S8k(Zlot+b+=)^f-<4xbUr;ho(4g`H|_W4Q*)zXPf)zauxDY~UZ8uq;IEzMoNMoy*j&w>-
*6^Ll_`@eZ&g^lkbjjR^lBA4G~vtL4sn^T+QPu~oCtOs_b3SlCwvws5(y9D4zn!XtY)}OSchPK0tDR92=zM}!&#Qhi&n8*)AeGuj
?hQRS0p@nBCnU_^uJwf6OUUY(Sp*85LR1a{n-SK*z{d%R;2w$R;@b%!iAR63MD*VLe@Hwdvz+~Z*@T0DFsZ-
>{SjyOiq2TJQXtO4^D+Nug%FTj<Vohqqz<jqv4&$056i3tG*quT&aC`fH&cCvRcuih{`FhWx~`HGq}$TK_OGE``%g}y+%RKaPS3&
Ac+@w%cdJp0!=(2(!@HHv7MJcbOr1f`SA?5`+5e^s^gU*-
F6=5Fp1a(2KHttk}mp`k&wZ7&SvFd3}gNg6h>#VFFBkpF!+y5GCGQZXFOZ??SPVawT5a>x9#Iy*3d>gYioHBMe4q`nVhf8DX3>!^
NzgH=(L(#P0I~P3<-pJFEUq2xcen6>;ib7+km-u?1h*K3wNY6Di49H?&*26s#zQNOx9R~1Fd6KG-
!Q@mzxEhe^{UL&1|wK=J6jUMA!MBN1pvrnZ$U?#W5f)8(X__^OscsE6b5p-<CslKvDS(We_=7JmGbk-
n7JyMCf}MD0A+$v`M1qrQSu`ZwmzeS$dtM&Oj`oh*3Po_q<V1r)09r%{_M;uXjT3l4O%>9GN06Rn}i2RI0LDNh$4AWh{@_m10Oc+
hpfH1j+%cUW!FN5ze((8h;vI1d5TlkW6Srsh%gEcKX`}>U-*hcWryJ?myxz4fW?#&3H-~3Q-
jKUh4}U#V~_;FnmV<6Vj&5om4;D_VG3&eE=Q#Xq8>^y)OWuCHq2WKh9leN&=RT32ofqSF;c`Vjgj~@)pRA$bD0scXo*1+G+Po>Hq
?VrSqf*iwyb{IOcq2p<r}0h>aU)Q5tL&410?M1}ox=#73mqxbZ4lENmv<vPo_A(S+O9$vCv+5I=42L~%(bBD0<WPs{Jwmd7u3oul
NIqBK20op)sXKA<mBXBZjiL=VcnlMJNJs&!1sIApqOR>gWsJ3~A9JG#t_wint6nrzTPg(O(l4_F=ae0CG$H_1Ag3#Sz>aJ6)i39!
dql+anv7G<^Gof=);*XT<v8Rh9}(yV5Eq?C_%YgT(7Sx=Xrqoh}f9}St$idN6&&dwFrfps*D@qa$1PQu;H^`eY+Xw$OV<+|0#(Co
H86*yk@3mU@A?O;oWt!^&>djtNo?UE!_Ys*;*!T?@X5gJ6dl8`=$9a7On<EM<qJ+So^j9^NRy;*^V<z$AS*BeNN2zrD6Cp_`P#LW
~RapCV3*D>~TkxY{UFM;VIXL!86r7On|Wf88Li>|<|0)1kl(J!absZ2wE0H25iTHk<CzJk~WhY}kxrI=O9Mw_?vbhH>M0perQL}W
KB>`J+I_p?GNSdQ<&aTuiNEW*naG2F#1sKgp!=CnZ_IpFs*RE?#FY{G`N-
&Cj+6Gnt}|E$W@c$`!DcvP#tCq2~v{2e5jsK(CgN+m5Fva8k>mS=z!h9AV;W5MqOO4%SRJb4&jZbp+CEAD&i7%96c#+~g~P^HIcw
gSXH-mcc_gKX}f(l=mmz==*t3{u{vxnrDK)U?l?6Is67yN51B5Ev)iWP2|xG}IB?vRgz@CF(^#_-LebP1FY6b1_-
m>tK+0i$S#&>k5-
u#p4UN<}qI!*!TP6Q^S#B9oGw!JVmO?aVKPh@{^W%;_xP4+ZuB4I>}zmm$L<dVn5#Yn(V>+!0O7+U9L56=>|ycz$eMzFvr^iOzzL
w)oOY3G$*X&LQzod9V+DGDlR9A^u3Wte^`6Lnv?*(Jbe4fvReCOxDxS)Wloi0TQJ4lCtv~m4@3L=y`Bi-
m4PJ^#gypb)2AW8yO2jCYsuzVj!+y>Zt?kxbL;fQ=)!yAjex53(-
0#}J;)jH1e8k@ya44xvRp6Xx6v}r*U?np(ii)lf|)hJ&&N$BfCl}sei}&}Kk7KQ9jX7mq3~3wnw&lJz`Hu?PMx-
;b9k#SW}s|8=vB(x=w)u)W$yGccY2xf$car}pLh~_l}V@^NEu!1lux3WPeBmyaqMy_rH!O!(Y!BQJhttEdshQjtbMhD8f{O|^H<S
t8WFt=#6n~S6aZ>;wwY@A(poiFGK9C<A#S`-
9Wz1ZAjjf=#CX?7UA3FXRl7T+8i%Mw`OQSTLKGyzv1Mjq=5M^jYEj9aZy&GgI5up9vVHBL)lGwk4iMW1FL={Dh*x(>mafotZ#93y
t;F!-iw_^em#?3GcqKL!n{AAOYxc#>Nn$Iwc2%HC?RArwd3h8HaC@2X{&Ckl?YnMp9Ez9Mo{MA`q+-P}a4-
>8H}S)oB$o7MKjOY8tKzl|%-
Xa2(w2MNiGLW)LK=K!8m|mSoXuKK=qRT;)M=P?SWjAY;&tWLfwDClQt6LWcntE&;p}EDR)X4r=~s}mtNq+-
@pu$s+NOf~R^~_9@ScW<wUjeD_1K${B^Gme<8SpwWUE<aTW_;p%u!GjtPQ4ORcYa`4*P1qb!^oLGu*n$T+-T-
hLdSu*}W~Tsj_NkSDin}(sDDl3^n^NGPO#}Pb;QHrkRt8UD#(T^(I9`vjyo;Fh*3?<k)&68d1*=v>eN*p>JiZ*cRs@HiT`7KC15B
;dKwNUF?baZ*Iy2Rc|G0D$>ihzWjEhzHN~|{ckCMs_Qv}K^*e&P?6Dhij48&M26YL{>gz%yiKn3zl~fOf0bP6*T|KzDOa|p(=k#U
RgYXc3hsF%umpVIK`ueE{V};EY9go{mW0Ks?)G;bc~Xsk>xS!(B-
M6wKeeE{=IaH7JGGwrX3)zP4}ooW^enq$?6x~JIz9JoIaON~%nwocqKp@^T>hf3I8^!dZfoibLuH!o7FV=nAd=FTxGi(j>t2;_po
Gg#9ij#m0v=Q8AZNjUcl{`19>6-J<ysqEI)id~A#$^UZll*`#MSnEV_%<ZOmt39)g+w&+%IQwnc&IDB-JEeRed_Ct=9%xfX&QNj?
__Bd6<&sMcu`AW+;%pniGBmUe9SiR1&&oB@n2dENb&$VT%{`+OM+wvmHjxV{uz4LY?vlD*DvB_*9=d?LyYBsodXQdBPf1ZrgB#xJ
m?ro&Y*ijB#(%bp0S=q+v;nz8q-gE)LV;I?g!JjM^C4#{r-
hG%ijVh;+z3|J3UoAD{Yz6Jzg2>t7bjy+(Av!QTR4Jm~%<+0C2>LvlhWB=0f4Iu&znIni*tokFM<AdJ1&$!c@Y7mtF0nPBd5ZUph
(q~P?f8V{F{0qJFj*)bDd#VIJ(jB(Fmft+nxK)UIR6hX&X@()SLP2s$yO(NesK_T5cn&NHoby<9@LYL8^6>7k~NgmHz_`9EB55X#
^bDbIoBX@g4t<#=g=@~UMs__IrOP7~0JAA<`Pm$Pc(_@a%b6UDgm7}!ojGg>`!OqmLr39ho1U;x-3Z8AxoNj1P@Tle`<#xeLx-wI
+yrAb#mF}<|t6*{hF9O!L`B`lVPyw~K!t)w6CoUSA*|F=W!pJ_V!082hbOh%3^u#weioc#Y^+4u$G0BUpe5)PkWA|&CEu&)6oNbo
Nho<(H7Yf<|r^;2&G}{wn>dP`F4NT0RS~JLt95)5X$>@#{JNse0LIz}4{yl-gb+A<kwG2}TEh2k^k;vlMAL3oKoiW%~gV7&>XrcT
##Ffr>0R`~XhnEkbSlGj;DV`UE^#TQ*@^zC%*Z&GoOUx<7V1`@3$dkFMs%W@VsiDs(meGG3#W|a^Hm_LPVt&0tC<aB;@e1e;1Z4~
wGBe85CoH0@USxnlE6umRynWH!v!<09Ou=~h>+9Et!!Xho;bC!3O?7GlBGk~-
XtYDEMvLTX)vPU@=gZJSX;UoJo3)W{B<9qyq^2I59}Ibhntqw2Z=zomif6Y~047q4ajM(Al}hy2%>UK07r$ndHJF_yP4JLFQS<dE
TjK2W=8WZn)olH`VQ1@lRivrP1`daQqxNPD-O5mI{&{2m6x%tv<-
f)17gcrRzFpn>rGu=_y<ZY=V?03H8IMo+R!t?|=o(*Xn949dVmOj$oFO6)S74Z81kwtF@yKM&TNK8@S&YuViP8EYF1(s5lRobBvi
VA}j#!KhS(%`c&DVO9vuPNYG89Nsl<%U-3qWh#j&f-
y8~ey~*x5oSsMcf_&jeaAZ32gJF&gD@%5MQn8)DU*+JCoN7&PSs`<}?xU|caV@bN!ksNg+?XFy>m@}9zPh@UdCkJcl+Q<9fvPxf{
4H}n(*`u=6t$nR`lJgVO}+SM2OoNgAlzC7euIh|Ot_%7J{gY<Y~!rQ(v(LaIt8D4C~4A`}Ja>KYX!-
O(A!L08)p$z2>4d#A1@5Gc7h!La}Qj)vpdrV+gle-k2ty6r%bSBE42m?ev2LSTk67~-6*Q9eb5OCX-AtGI}^X3rw&bUCNTkZF@yF
IpfYfA#X(Hyo0-
o8ykI(R4Mx8FA6l~K{?S{C1K6ybnS3KC0F1$B}{BQUj9tU4zU@AHhausS;?ZFn|DtN#y>Roi@cr9)DUejA=Gzq|8Thm>pGy@JMZ$
6!wy*bP!#!eYX!i9u9_aGLk*0rI%)wsH0KVZ(exxFw4X6=G@jw|wpdd#HS&(W_8$k+KdwtO^_O-Qh`9m?P;VBhs~yQyJU2-
8vlg&hXN}lm5szU7GfFWO%F?nZqy?71hgYZ;eC0J#px_;kgdF3=cTy!s;%AqUk0#;qZ3iy&%^z-
G;sCD%6pZk;6km1Pmd%tBr#o8%Y?m&8nr}y=7hi#%Xx?BnM3$Xil8EVgIY9PEohL^TbD1T4E`v%Pm<_KibmL@(SrYArhK)ecv__Y
2-j+_;QOwLXyAT^04dJpxY@h4r-
#^*bit0p(WmGz=}g6=!yazb=DhN6RVxTimoE`?T2ZWTqP@L49{H2^V8x+3^Rb$O0RoQ6U<i)ljR9ovbb+=y;;p^H!DxPZQu57yV*
E1PYI^dO*V}wxd(Njy~Tvy)yzRq4Tu-
jK_W?~>ligKASKYXFg?GqpIfQdrfC+Jaw>+I+ALem%i?<GU*^}d?hlvObjQ$bsZ4I<S3J0V+W5a*>05g
'''
patch = zlib.decompress(base64.b85decode("".join(encoded_patch.split())))
assert hashlib.sha256(patch).hexdigest() == EXPECTED_PATCH_SHA256, "内嵌补丁校验失败"
patch_path.write_bytes(patch)
run(["git", "apply", str(patch_path)], "apply-local-fp8-patch")
actual_patch = subprocess.check_output(["git", "diff", "--binary"], cwd=root)
assert actual_patch == patch, "远端补丁与本地修改不一致"
run(["git", "diff", "--check"], "source-diff-check")
run(["git", "status", "--short"], "source-status")
run(["git", "diff", "--stat"], "source-diff-stat")
os.chdir(root)

run(["nvidia-smi"], "gpu")
run(["nvcc", "--version"], "cuda-toolkit")


In [ ]:
run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], "install-uv")
uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
run(["bash", "-lc", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"],
    "install-rust")
os.environ["PATH"] = str(Path.home() / ".cargo/bin") + os.pathsep + os.environ["PATH"]

run([uv, "venv", "--python", "3.12", ".venv"], "venv")
py = str(root / ".venv/bin/python")
run([uv, "pip", "install", "--python", py, "-r", "requirements/build/cuda.txt",
     "--torch-backend=auto"], "build-dependencies")
run([uv, "pip", "install", "--python", py, "-r", "requirements/rwkv.txt",
     "--torch-backend=auto"], "runtime-dependencies")
run([uv, "pip", "install", "--python", py, "-e", ".", "--no-build-isolation",
     "--torch-backend=auto"], "build-irwkv", {"MAX_JOBS": "2"})
run([uv, "pip", "install", "--python", py, "pytest"], "test-dependencies")
run([uv, "pip", "freeze", "--python", py], "packages")


In [ ]:
gpu_probe = r'''
import json
from pathlib import Path
import torch
from vllm.vllm_flash_attn.flash_attn_interface import is_fa_version_supported
from vllm.v1.attention.ops.triton_unified_attention import unified_attention

assert torch.cuda.is_available(), "CUDA unavailable"
capability = torch.cuda.get_device_capability()
properties = torch.cuda.get_device_properties(0)
support = {}
for version in (2, 3, 4):
    try:
        support[str(version)] = bool(is_fa_version_supported(version))
    except Exception as error:
        support[str(version)] = f"{type(error).__name__}: {error}"
info = {
    "device_name": torch.cuda.get_device_name(0),
    "capability": list(capability),
    "total_memory_gib": properties.total_memory / 2**30,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "fa_version_supported": support,
    "triton_unified_attention": callable(unified_attention),
}
Path("/content/irwkv-fp8-validation-logs/gpu-capability.json").write_text(
    json.dumps(info, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(info, indent=2))
assert capability >= (8, 9), f"FP8 W8A8 requires SM89+, got {capability}"
assert info["bf16_supported"], "BF16 unavailable"
'''
run([py, "-c", gpu_probe], "gpu-capability")


In [ ]:
run([py, "-m", "pytest", "--noconftest", "tests/models/rwkv/test_rwkv7_hybrid.py",
     "-q", "-rs", "-k",
     "cuda_fp8_scaled_mm_matches_bux_emulation or attention_prepare_decode_matches_strided_eager_views_and_pages"],
    "targeted-fp8-cuda", {"IRWKV_TEST_FLASH_HYBRID": "1"})


In [ ]:
run([py, "-m", "pytest", "--noconftest", "tests/models/rwkv/test_rwkv7_hybrid.py", "-q", "-rs"],
    "full-hybrid-correctness", {"IRWKV_TEST_FLASH_HYBRID": "1"})


In [ ]:
run([py, "-m", "tools.export_rwkv7_hybrid", "exported/hybrid-smoke",
     "--width", "64", "--layers", "4", "--vocab-size", "4096"], "export-smoke")

common_args = [
    "exported/hybrid-smoke", "--concurrency", "2", "--sessions", "2",
    "--system", "64", "--context", "128", "--tool", "32", "--output", "8",
    "--turns", "4", "--max-model-len", "1024", "--max-num-batched-tokens", "128",
]
run([py, "-m", "benchmarks.benchmark_rwkv7_hybrid_agentic", *common_args,
     "--tag", "colab-g4-bf16-smoke"], "engine-bf16-smoke")
run([py, "-m", "benchmarks.benchmark_rwkv7_hybrid_agentic", *common_args,
     "--quantization", "fp8", "--kv-cache-dtype", "auto",
     "--tag", "colab-g4-fp8-w8a8-auto-kv-smoke"], "engine-fp8-weights-auto-kv-smoke")


In [ ]:
import json
gpu_info = json.loads((logs / "gpu-capability.json").read_text(encoding="utf-8"))
major, minor = gpu_info["capability"]
backend = {9: "fa3", 10: "fa4", 12: "triton"}.get(major)
combined_status = ""

if backend is not None:
    run([py, "-m", "benchmarks.benchmark_rwkv7_hybrid_agentic", *common_args,
         "--quantization", "fp8", "--kv-cache-dtype", "fp8",
         "--tag", f"colab-{backend}-fp8-w8a8-fp8-kv-smoke"],
        "engine-fp8-weights-fp8-kv-smoke")
    combined_status = f"passed with {backend}"
else:
    rejection_probe = r'''
from irwkv.engine.arg_utils import EngineArgs
try:
    EngineArgs(
        model="exported/hybrid-smoke",
        load_format="dummy",
        dtype="bfloat16",
        quantization="fp8",
        kv_cache_dtype="fp8",
        skip_tokenizer_init=True,
        enable_prefix_caching=True,
        max_model_len=1024,
        max_num_batched_tokens=128,
    ).create_engine_config()
except ValueError as error:
    assert "SM90, SM100, or SM120" in str(error), str(error)
    print("Expected rejection:", error)
else:
    raise AssertionError("Combined FP8 KV was unexpectedly accepted on this device")
'''
    run([py, "-c", rejection_probe], "engine-fp8-kv-expected-rejection")
    combined_status = f"not run: unsupported SM{major}{minor}"

completion = {
    "base": BASE,
    "patch_sha256": EXPECTED_PATCH_SHA256,
    "gpu": gpu_info,
    "targeted_fp8_cuda": "passed",
    "full_hybrid_correctness": "passed; inspect pytest skip counts in log",
    "engine_bf16_smoke": "passed",
    "engine_fp8_weights_auto_kv_smoke": "passed",
    "engine_fp8_weights_fp8_kv_smoke": combined_status,
}
(logs / "completion.json").write_text(json.dumps(completion, indent=2, ensure_ascii=False) + "\n",
                                       encoding="utf-8")
print(json.dumps(completion, indent=2, ensure_ascii=False))


## 下载结果

无论上面全部通过还是中途失败，都运行下面这个单元，并把下载的 ZIP 发回。
若中途失败，ZIP 里会保留失败前的全部日志；不要只发最后一行错误。


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

logs = Path("/content/irwkv-fp8-validation-logs")
assert logs.exists(), "没有找到日志目录；请从第一个单元开始运行"
archive = shutil.make_archive("/content/irwkv-fp8-validation-logs", "zip", logs)
print(archive)
files.download(archive)
